<a href="https://colab.research.google.com/github/thanita47/text2sql-qlora-finetuning/blob/main/text2sql_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text2SQL Fine-tuning: Qwen2.5-Coder-1.5B-Instruct with QLoRA

## 1. Setup

In [ ]:
!nvidia-smi

Fri Jul 17 05:44:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#LLaMA-Factory
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,bitsandbytes]" --quiet

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 27711, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 27711 (delta 41), reused 22 (delta 22), pack-reused 27618 (from 2)
Receiving objects: 100% (27711/27711), 13.59 MiB | 9.49 MiB/s, done.
Resolving deltas: 100% (19773/19773), done.
/content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip show llamafactory

Name: llamafactory
Version: 0.9.6.dev0
Summary: Unified Efficient Fine-Tuning of 100+ LLMs
Home-page: https://github.com/hiyouga/LLaMA-Factory
Author: 
Author-email: hiyouga <hiyouga@buaa.edu.cn>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Editable project location: /content/LLaMA-Factory
Requires: accelerate, av, datasets, einops, fastapi, fire, gradio, hf-transfer, matplotlib, modelscope, numpy, omegaconf, packaging, pandas, peft, protobuf, pydantic, pyyaml, safetensors, scipy, sentencepiece, sse-starlette, tiktoken, torch, torchaudio, torchdata, torchvision, transformers, trl, tyro, uvicorn
Required-by: 


In [ ]:
%cd /content/LLaMA-Factory
import llamafactory
print("LlamaFactory installed successfully")

/content/LLaMA-Factory
LlamaFactory installed successfully


## 2. Dataset (Round 1: Spider) — Load & EDA

In [ ]:
from datasets import load_dataset

# ใช้ namespace เต็มของ Spider dataset
dataset = load_dataset("xlangai/spider")
print(dataset)
print(dataset['train'][0])

README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

spider/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  831kB            

spider/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

spider/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  126kB            

spider/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})
{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']}


In [ ]:
import pandas as pd

# แปลงเป็น DataFrame เพื่อดูง่ายๆ
df = pd.DataFrame(dataset['train'])

print(f"จำนวนตัวอย่าง (train): {len(df)}")
print(f"จำนวน database ที่ต่างกัน: {df['db_id'].nunique()}")
print(f"\nColumns ที่มี: {df.columns.tolist()}")
print(f"\nตัวอย่าง 3 แถวแรก:")
df.head(3)

จำนวนตัวอย่าง (train): 7000
จำนวน database ที่ต่างกัน: 140

Columns ที่มี: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks']

ตัวอย่าง 3 แถวแรก:


,db_id,query,question,query_toks,query_toks_no_value,question_toks
0,department_management,SELECT count(*) FROM head WHERE age > 56,How many heads of the departments are older th...,"[SELECT, count, (, *, ), FROM, head, WHERE, ag...","[select, count, (, *, ), from, head, where, ag...","[How, many, heads, of, the, departments, are, ..."
1,department_management,"SELECT name , born_state , age FROM head ORD...","List the name, born state and age of the heads...","[SELECT, name, ,, born_state, ,, age, FROM, he...","[select, name, ,, born_state, ,, age, from, he...","[List, the, name, ,, born, state, and, age, of..."
2,department_management,"SELECT creation , name , budget_in_billions ...","List the creation year, name and budget of eac...","[SELECT, creation, ,, name, ,, budget_in_billi...","[select, creation, ,, name, ,, budget_in_billi...","[List, the, creation, year, ,, name, and, budg..."


In [ ]:
# วิเคราะห์ความซับซ้อนของ query
df['has_join'] = df['query'].str.contains('JOIN', case=False)
df['has_groupby'] = df['query'].str.contains('GROUP BY', case=False)
df['has_nested'] = df['query'].str.contains(r'\(SELECT', case=False, regex=True)
df['has_orderby'] = df['query'].str.contains('ORDER BY', case=False)
df['query_length'] = df['query'].str.len()

print(f"Query ที่มี JOIN: {df['has_join'].mean()*100:.1f}%")
print(f"Query ที่มี GROUP BY: {df['has_groupby'].mean()*100:.1f}%")
print(f"Query ที่ nested (subquery): {df['has_nested'].mean()*100:.1f}%")
print(f"Query ที่มี ORDER BY: {df['has_orderby'].mean()*100:.1f}%")
print(f"\nความยาว query เฉลี่ย: {df['query_length'].mean():.0f} ตัวอักษร")
print(f"ความยาว query สูงสุด: {df['query_length'].max()} ตัวอักษร")

# ดู database ที่มีตัวอย่างเยอะสุด 5 อันดับแรก
print(f"\nDatabase ที่มีตัวอย่างเยอะสุด 5 อันดับ:")
print(df['db_id'].value_counts().head())

Query ที่มี JOIN: 39.8%
Query ที่มี GROUP BY: 25.4%
Query ที่ nested (subquery): 6.5%
Query ที่มี ORDER BY: 23.3%

ความยาว query เฉลี่ย: 110 ตัวอักษร
ความยาว query สูงสุด: 577 ตัวอักษร

Database ที่มีตัวอย่างเยอะสุด 5 อันดับ:
db_id
college_2    170
college_1    164
hr_1         124
store_1      112
soccer_2     106
Name: count, dtype: int64


## 3. Prepare Instruction Format (v1 — no schema)

In [ ]:
import json

def format_instruction(example):
    return {
        "instruction": f"Given the database schema for '{example['db_id']}', translate this question into a SQL query: {example['question']}",
        "input": "",
        "output": example['query']
    }

# แปลงทั้ง train set
formatted_data = [format_instruction(row) for row in dataset['train']]

# ดูตัวอย่างก่อนเซฟ
print(json.dumps(formatted_data[0], indent=2, ensure_ascii=False))
print(f"\nจำนวนตัวอย่างทั้งหมด: {len(formatted_data)}")

# เซฟเป็นไฟล์ JSON
with open("train_data.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, indent=2, ensure_ascii=False)

print("บันทึกไฟล์ train_data.json เรียบร้อย")

{
  "instruction": "Given the database schema for 'department_management', translate this question into a SQL query: How many heads of the departments are older than 56 ?",
  "input": "",
  "output": "SELECT count(*) FROM head WHERE age  >  56"
}

จำนวนตัวอย่างทั้งหมด: 7000
บันทึกไฟล์ train_data.json เรียบร้อย


In [ ]:
# แปลง validation set ด้วย format เดียวกัน
val_formatted_data = [format_instruction(row) for row in dataset['validation']]

# ดูตัวอย่างก่อนเซฟ
print(json.dumps(val_formatted_data[0], indent=2, ensure_ascii=False))
print(f"\nจำนวนตัวอย่าง validation ทั้งหมด: {len(val_formatted_data)}")

# เซฟเป็นไฟล์ JSON
with open("val_data.json", "w", encoding="utf-8") as f:
    json.dump(val_formatted_data, f, indent=2, ensure_ascii=False)

print("บันทึกไฟล์ val_data.json เรียบร้อย")

{
  "instruction": "Given the database schema for 'concert_singer', translate this question into a SQL query: How many singers do we have?",
  "input": "",
  "output": "SELECT count(*) FROM singer"
}

จำนวนตัวอย่าง validation ทั้งหมด: 1034
บันทึกไฟล์ val_data.json เรียบร้อย


In [ ]:
import json, os

# path ของ dataset_info.json ใน LLaMA-Factory
dataset_info_path = "/content/LLaMA-Factory/data/dataset_info.json"

# copy ไฟล์ train/val ของเราไปไว้ในโฟลเดอร์ data/ ของ LLaMA-Factory
import shutil
shutil.copy("train_data.json", "/content/LLaMA-Factory/data/train_data.json")
shutil.copy("val_data.json", "/content/LLaMA-Factory/data/val_data.json")

# โหลด dataset_info.json เดิม
with open(dataset_info_path, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

# เพิ่ม entry ของเราเข้าไป
dataset_info["text2sql_thai_train"] = {
    "file_name": "train_data.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

dataset_info["text2sql_thai_val"] = {
    "file_name": "val_data.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

# เซฟกลับ
with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print("ลงทะเบียน dataset กับ LLaMA-Factory เรียบร้อย")
print("Dataset keys ที่เพิ่ม: text2sql_thai_train, text2sql_thai_val")

ลงทะเบียน dataset กับ LLaMA-Factory เรียบร้อย
Dataset keys ที่เพิ่ม: text2sql_thai_train, text2sql_thai_val


## 4. Train v1

In [ ]:
config_content = """
### model
model_name_or_path: Qwen/Qwen2.5-Coder-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
quantization_bit: 4
quantization_method: bitsandbytes

### dataset
dataset: text2sql_thai_train
eval_dataset: text2sql_thai_val
template: qwen
cutoff_len: 1024
max_samples: 7000
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: saves/qwen2.5-coder-text2sql/lora_r16
logging_steps: 10
save_steps: 500
save_total_limit: 1
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 2.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 4
eval_strategy: steps
eval_steps: 200
"""

with open("/content/LLaMA-Factory/qwen_lora_sft.yaml", "w", encoding="utf-8") as f:
    f.write(config_content)

print("บันทึกไฟล์ qwen_lora_sft.yaml เรียบร้อย")
print(config_content)

บันทึกไฟล์ qwen_lora_sft.yaml เรียบร้อย

### model
model_name_or_path: Qwen/Qwen2.5-Coder-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
quantization_bit: 4
quantization_method: bitsandbytes

### dataset
dataset: text2sql_thai_train
eval_dataset: text2sql_thai_val
template: qwen
cutoff_len: 1024
max_samples: 7000
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: saves/qwen2.5-coder-text2sql/lora_r16
logging_steps: 10
save_steps: 500
save_total_limit: 1
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 2.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 4
eval_strategy: steps
eval_steps: 200



In [ ]:
%cd /content/LLaMA-Factory
!llamafactory-cli train qwen_lora_sft.yaml

/content/LLaMA-Factory
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[WARNING|2026-07-17 05:47:13] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-07-17 05:47:13] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 660/660 [00:00<00:00, 2.81MB/s]
[INFO|configuration_utils.p

## 5. Evaluate v1 — Exact-Match Accuracy

In [ ]:
predict_config = """
### model
model_name_or_path: Qwen/Qwen2.5-Coder-1.5B-Instruct
adapter_name_or_path: saves/qwen2.5-coder-text2sql/lora_r16
trust_remote_code: true

### method
stage: sft
do_predict: true
finetuning_type: lora

### dataset
eval_dataset: text2sql_thai_val
template: qwen
cutoff_len: 1024
max_samples: 1034
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: saves/qwen2.5-coder-text2sql/predict_r16
overwrite_output_dir: true

### eval
per_device_eval_batch_size: 4
predict_with_generate: true
"""

with open("/content/LLaMA-Factory/predict_config.yaml", "w", encoding="utf-8") as f:
    f.write(predict_config)

print("บันทึกไฟล์ predict_config.yaml เรียบร้อย")

บันทึกไฟล์ predict_config.yaml เรียบร้อย


In [ ]:
!pip install rouge_chinese nltk jieba --quiet

In [ ]:
%cd /content/LLaMA-Factory
!llamafactory-cli train predict_config.yaml

/content/LLaMA-Factory
[INFO|2026-07-17 06:51:30] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: None
[INFO|configuration_utils.py:780] 2026-07-17 06:51:30,553 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/2e1fd397ee46e1388853d2af2c993145b0f1098a/config.json
[INFO|configuration_utils.py:856] 2026-07-17 06:51:30,560 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

In [ ]:
import json

results = []
with open(f"{path}/generated_predictions.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        results.append(json.loads(line))

print(f"จำนวนตัวอย่างที่ predict ทั้งหมด: {len(results)}")

# Normalize SQL ก่อนเทียบ (ตัด whitespace, newline, lowercase)
def normalize_sql(sql):
    return " ".join(sql.strip().lower().replace("\n", " ").split())

correct = 0
fail_examples = []

for r in results:
    pred = normalize_sql(r['predict'])
    label = normalize_sql(r['label'])
    if pred == label:
        correct += 1
    else:
        fail_examples.append({
            "prompt": r['prompt'],
            "predict": r['predict'],
            "label": r['label']
        })

accuracy = correct / len(results) * 100
print(f"\n{'='*50}")
print(f"Exact-Match Accuracy: {accuracy:.2f}% ({correct}/{len(results)})")
print(f"จำนวนตัวอย่างที่ fail: {len(fail_examples)}")
print(f"{'='*50}")

# เซฟผลลัพธ์ไว้ใช้ต่อ (error analysis + README)
with open("eval_summary.json", "w", encoding="utf-8") as f:
    json.dump({
        "total": len(results),
        "correct": correct,
        "accuracy": accuracy,
        "fail_examples_sample": fail_examples[:20]  # เก็บตัวอย่าง fail ไว้ 20 อัน
    }, f, indent=2, ensure_ascii=False)

print("\nบันทึกไฟล์ eval_summary.json เรียบร้อย")

จำนวนตัวอย่างที่ predict ทั้งหมด: 1034

Exact-Match Accuracy: 11.70% (121/1034)
จำนวนตัวอย่างที่ fail: 913

บันทึกไฟล์ eval_summary.json เรียบร้อย


## 6. Error Analysis — Diagnosing the Schema Hallucination Issue

In [ ]:
for i, ex in enumerate(fail_examples[:5]):
    print(f"=== ตัวอย่างที่ {i+1} ===")
    print(f"Predict: {ex['predict']}")
    print(f"Label:   {ex['label']}")
    print()

=== ตัวอย่างที่ 1 ===
Predict: SELECT T2.name ,  T2.release_year FROM concert AS T1 JOIN song AS T2 ON T1.song_id  =  T2.song_id JOIN band AS T3 ON T3.band_id  =  T1.band_id ORDER BY T3.age LIMIT 1
Label:   SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1


=== ตัวอย่างที่ 2 ===
Predict: SELECT T2.name ,  T2.release_year FROM concert AS T1 JOIN song AS T2 ON T1.song_id  =  T2.song_id JOIN singer AS T3 ON T3.singer_id  =  T1.singer_id ORDER BY T3.age LIMIT 1
Label:   SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1


=== ตัวอย่างที่ 3 ===
Predict: SELECT DISTINCT T1.country FROM people AS T1 JOIN band AS T2 ON T1.people_id  =  T2.people_id WHERE T1.age  >  20
Label:   SELECT DISTINCT country FROM singer WHERE age  >  20


=== ตัวอย่างที่ 4 ===
Predict: SELECT T2.song_name FROM singer AS T1 JOIN song AS T2 ON T1.singer_id  =  T2.singer_id WHERE T1.age  >  (SELECT avg(age) FROM singer)
Label:   SELECT song_name FROM singer WHERE age  >  (SELECT 

## 7. Dataset (Round 2: sql-create-context) — Schema-Aware

In [ ]:
from datasets import load_dataset

sql_dataset = load_dataset("b-mc2/sql-create-context")
print(sql_dataset)
print(sql_dataset['train'][0])

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})
{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


## 8. Prepare Instruction Format (v2 — with schema) & Train

In [ ]:
import json

def format_instruction_with_schema(example):
    return {
        "instruction": f"Given the database schema: {example['context']} Translate this question into a SQL query: {example['question']}",
        "input": "",
        "output": example['answer']
    }

# ใช้แค่บางส่วนเพื่อความเร็ว (5000 ตัวอย่างพอ ไม่ต้องเอาทั้งหมด)
train_subset = sql_dataset['train'].select(range(5000))
formatted_data_v2 = [format_instruction_with_schema(row) for row in train_subset]

with open("train_data_v2.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data_v2, f, indent=2, ensure_ascii=False)

print(f"จำนวนตัวอย่าง: {len(formatted_data_v2)}")
print(json.dumps(formatted_data_v2[0], indent=2, ensure_ascii=False))

จำนวนตัวอย่าง: 5000
{
  "instruction": "Given the database schema: CREATE TABLE head (age INTEGER) Translate this question into a SQL query: How many heads of the departments are older than 56 ?",
  "input": "",
  "output": "SELECT COUNT(*) FROM head WHERE age > 56"
}


In [ ]:
val_subset = sql_dataset['train'].select(range(5000, 5500))  # เอา 500 ตัวอย่างถัดมาเป็น val
val_formatted_data_v2 = [format_instruction_with_schema(row) for row in val_subset]

with open("val_data_v2.json", "w", encoding="utf-8") as f:
    json.dump(val_formatted_data_v2, f, indent=2, ensure_ascii=False)

print(f"จำนวนตัวอย่าง validation: {len(val_formatted_data_v2)}")

จำนวนตัวอย่าง validation: 500


In [ ]:
config_content_v2 = """
### model
model_name_or_path: Qwen/Qwen2.5-Coder-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
quantization_bit: 4
quantization_method: bitsandbytes

### dataset
dataset: text2sql_schema_train
eval_dataset: text2sql_schema_val
template: qwen
cutoff_len: 1024
max_samples: 5000
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: saves/qwen2.5-coder-text2sql/lora_schema_v2
logging_steps: 10
save_steps: 500
save_total_limit: 1
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 2.0e-4
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 4
eval_strategy: steps
eval_steps: 200
"""

with open("/content/LLaMA-Factory/qwen_lora_sft_v2.yaml", "w", encoding="utf-8") as f:
    f.write(config_content_v2)

print("บันทึก config v2 เรียบร้อย")

บันทึก config v2 เรียบร้อย


In [ ]:
%cd /content/LLaMA-Factory
!llamafactory-cli train qwen_lora_sft_v2.yaml

/content/LLaMA-Factory
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[WARNING|2026-07-17 07:29:35] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-07-17 07:29:35] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:780] 2026-07-17 07:29:35,792 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/2e1fd397ee46e1388853d2af2c993145b0f1098a/config.json
[INFO|configuration_utils.py:856] 2026-07-17 07:29:35,800 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_siz

## 9. Evaluate v2 — Results Comparison

In [ ]:
predict_config_v2 = """
### model
model_name_or_path: Qwen/Qwen2.5-Coder-1.5B-Instruct
adapter_name_or_path: saves/qwen2.5-coder-text2sql/lora_schema_v2
trust_remote_code: true

### method
stage: sft
do_predict: true
finetuning_type: lora

### dataset
eval_dataset: text2sql_schema_val
template: qwen
cutoff_len: 1024
max_samples: 500
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: saves/qwen2.5-coder-text2sql/predict_schema_v2
overwrite_output_dir: true

### eval
per_device_eval_batch_size: 4
predict_with_generate: true
"""

with open("/content/LLaMA-Factory/predict_config_v2.yaml", "w", encoding="utf-8") as f:
    f.write(predict_config_v2)

print("บันทึกไฟล์ predict_config_v2.yaml เรียบร้อย")

บันทึกไฟล์ predict_config_v2.yaml เรียบร้อย


In [ ]:
%cd /content/LLaMA-Factory
!llamafactory-cli train predict_config_v2.yaml

/content/LLaMA-Factory
[INFO|2026-07-17 07:59:47] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: None
[INFO|configuration_utils.py:780] 2026-07-17 07:59:48,019 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/2e1fd397ee46e1388853d2af2c993145b0f1098a/config.json
[INFO|configuration_utils.py:856] 2026-07-17 07:59:48,026 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

In [ ]:
import json

path_v2 = "/content/LLaMA-Factory/saves/qwen2.5-coder-text2sql/predict_schema_v2"

results_v2 = []
with open(f"{path_v2}/generated_predictions.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        results_v2.append(json.loads(line))

print(f"จำนวนตัวอย่างที่ predict: {len(results_v2)}")

def normalize_sql(sql):
    return " ".join(sql.strip().lower().replace("\n", " ").split())

correct_v2 = 0
fail_examples_v2 = []

for r in results_v2:
    pred = normalize_sql(r['predict'])
    label = normalize_sql(r['label'])
    if pred == label:
        correct_v2 += 1
    else:
        fail_examples_v2.append(r)

accuracy_v2 = correct_v2 / len(results_v2) * 100
print(f"\n{'='*50}")
print(f"Exact-Match Accuracy (v2 - with schema): {accuracy_v2:.2f}% ({correct_v2}/{len(results_v2)})")
print(f"เทียบกับ v1 (no schema): 11.70%")
print(f"{'='*50}")

with open("eval_summary_v2.json", "w", encoding="utf-8") as f:
    json.dump({
        "total": len(results_v2),
        "correct": correct_v2,
        "accuracy": accuracy_v2,
        "fail_examples_sample": fail_examples_v2[:20]
    }, f, indent=2, ensure_ascii=False)

print("บันทึกไฟล์ eval_summary_v2.json เรียบร้อย")

จำนวนตัวอย่างที่ predict: 500

Exact-Match Accuracy (v2 - with schema): 72.80% (364/500)
เทียบกับ v1 (no schema): 11.70%
บันทึกไฟล์ eval_summary_v2.json เรียบร้อย
